# LC 424 — Longest Repeating Character Replacement
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Sliding Window
**Pattern:** Valid Window = Size - Max Frequency ≤ k

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> The number of characters
you need to replace in a window equals the window
size minus the count of the most frequent character.
If that value exceeds k, shrink from the left.
</div>

## Official Problem Statement

You are given a string `s` and an integer `k`. You
can choose any character of the string and change
it to any other uppercase English character. You
can perform this operation at most `k` times.

Return the length of the longest substring containing
the same letter you can get after performing the
above operations.

**Example 1:**
```
Input:  s = "ABAB", k = 2
Output: 4
Explanation: Replace the two 'A's with 'B's or
             vice versa.
```
**Example 2:**
```
Input:  s = "AABABBA", k = 1
Output: 4
Explanation: Replace 'B' at index 5 -> "AABAAAA".
             Longest is 4.
```

**Constraints:**
- `1 <= s.length <= 10^5`
- `s` consists of only uppercase English letters
- `0 <= k <= s.length`

## What This Is Actually Asking

You can change up to k characters to anything you
want.
Find the longest contiguous block of the same
letter you can create by making at most k swaps.
You pick which character to dominate — the algorithm
figures out the best choice automatically.

## Walk Through an Example by Hand

```
s = "AABABBA"   k = 1
     0123456

count={} left=0 max_count=0 result=0

r=0 'A' count={A:1} max_count=1
    replacements = 1-1=0 <= k=1  valid  result=1

r=1 'A' count={A:2} max_count=2
    replacements = 2-2=0 <= 1    valid  result=2

r=2 'B' count={A:2,B:1} max_count=2
    replacements = 3-2=1 <= 1    valid  result=3

r=3 'A' count={A:3,B:1} max_count=3
    replacements = 4-3=1 <= 1    valid  result=4

r=4 'B' count={A:3,B:2} max_count=3
    replacements = 5-3=2 > k=1   INVALID
    shrink: s[left=0]='A' count={A:2,B:2}  left=1
    replacements = 4-2=2  still invalid? max_count=3
    (Note: max_count never decreases — this is intentional)
    window size=4, max_count=3: 4-3=1<=1  valid  result=4

r=5 'B' count={A:2,B:3} max_count=3
    replacements = 5-3=2 > 1     INVALID
    shrink: s[left=1]='A' count={A:1,B:3}  left=2
    result=4

r=6 'A' count={A:2,B:3} max_count=3
    replacements = 5-3=2 > 1     INVALID
    shrink: s[left=2]='B' count={A:2,B:2}  left=3
    result=4

Answer: 4
```

## The Picture

```
s = "A A B A B B A"   k = 1
     0 1 2 3 4 5 6

Window validity rule:

  replacements_needed = window_size - max_freq_char
  valid if replacements_needed <= k

Why? The dominant char stays; replace everything else.

  Window [A A B A]  size=4  max_freq(A)=3
  replacements = 4-3 = 1 <= k=1   VALID  len=4

  Window [A A B A B]  size=5  max_freq(A)=3
  replacements = 5-3 = 2 > k=1    INVALID -> shrink

Key trick: max_count NEVER decreases.
  We only care about windows LARGER than what we've
  already found — shrinking never gives a better
  answer, so we just slide the window forward.
```

## When To Use This Pattern

- When you see **at most k changes to make a window
  uniform**, think **size - max_freq <= k**
- When the window validity depends on frequency of
  the most common element, think **track max_count
  and never decrease it**
- When shrinking the window can never improve the
  answer, think **slide — don't actually shrink**
- When characters are a bounded alphabet (26 letters),
  think **O(1) space for the count map**

## The Approach

Maintain a frequency count of characters in the
current window and a running max_count of the most
frequent character ever seen in any window.
At each step, if the number of characters to replace
(window size minus max_count) exceeds k, shrink the
window by moving left one step.
Never decrease max_count — we only want windows
strictly larger than the current best.

In [1]:
# No extra imports needed — plain dict and variables

In [2]:
def test_harness(func):
    tests = [
        # (s, k, expected)
        ("ABAB",    2, 4),
        ("AABABBA", 1, 4),
        ("A",       0, 1),   # single char
        ("AAAA",    2, 4),   # already uniform
        ("ABCD",    0, 1),   # k=0 no replacements
        ("ABCD",    3, 4),   # k covers all but 1
        ("ABCDE",   1, 2),
        ("AABABBA", 0, 2),   # k=0 longest run of same
        ("KRSCDCSONAJNHLBMDQGIFCPEKPOHQIHLTDIQGEKLRLCQNBOHNDQGHJPNDQPERNFSSSRDEQLFPCCCARFMDLSASAFPCPN", 4, 7),
    ]

    passed = 0
    for i, (s, k, expected) in enumerate(tests):
        result = func(s, k)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        short_s = s[:20] + ('...' if len(s) > 20 else '')
        print(
            f"Test {i+1}: {status} | "
            f"s={short_s!r} k={k} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [6]:
def characterReplacement(s: str, k: int) -> int:
    """
    Return longest substring of one char after <= k
    replacements.

    Sliding window: track char frequencies and the
    running max_count (never decremented). If
    window_size - max_count > k, slide left one step.
    Window size only ever grows -> result = final size.

    Time:  O(n) — each char enters and exits once
    Space: O(1) — at most 26 keys in count dict
    """
    """
        Seans way: -
        keep a map of character occurances as they stream in the string.
        max character occurance in a string currently is max(hasmap.values())
        working with r on the main loop . l starts from a 0
        winSize = r-l+1
        when you find out that  winSize > max(hasmap.values()) -k ; start moving the left pointer
        in a while loop until the condition is no longer true.. (in loop decrement count of evicted char)

        after the while loop maintain res = max(res, winsize)
        var needed == hasmap and res
    """
    hashmap , l , res = {}, 0, 0
    for r, c in enumerate(s):
        #safely increment the count of c in hashmap
        hashmap[c] = hashmap.get(c, 0) + 1
        #sanitize the window making it valid after replacments by shrinking window left
        while (r - l + 1) > k + max(hashmap.values()) :  
            hashmap[s[l]] -= 1
            l+=1
        res = max(res, r-l+1) #since window is valid res is max of res and window size
    return res
            


# Quick debug — run this cell while building
print(characterReplacement("ABAB",    2))  # 4
print(characterReplacement("AABABBA", 1))  # 4
print(characterReplacement("ABCD",    0))  # 1
print(characterReplacement("AAAA",    2))  # 4
test_harness(characterReplacement)

4
4
1
4
Test 1: PASSED | s='ABAB' k=2 | expected=4 | got=4
Test 2: PASSED | s='AABABBA' k=1 | expected=4 | got=4
Test 3: PASSED | s='A' k=0 | expected=1 | got=1
Test 4: PASSED | s='AAAA' k=2 | expected=4 | got=4
Test 5: PASSED | s='ABCD' k=0 | expected=1 | got=1
Test 6: PASSED | s='ABCD' k=3 | expected=4 | got=4
Test 7: PASSED | s='ABCDE' k=1 | expected=2 | got=2
Test 8: PASSED | s='AABABBA' k=0 | expected=2 | got=2
Test 9: PASSED | s='KRSCDCSONAJNHLBMDQGI...' k=4 | expected=7 | got=7

9/9 tests passed


In [5]:
def characterReplacement(s: str, k: int) -> int:
    hashmap = {}
    l = 0
    res = 0
    max_f = 0  # Track the highest frequency seen in the current window
    
    for r, c in enumerate(s):
        hashmap[c] = hashmap.get(c, 0) + 1
        max_f = max(max_f, hashmap[c])
        
        # If replacements needed (size - max_f) > k, shrink window
        if (r - l + 1) - max_f > k:
            hashmap[s[l]] -= 1
            l += 1
            
        res = max(res, r - l + 1)
        
    return res
# Quick debug — run this cell while building
print(characterReplacement("ABAB",    2))  # 4
print(characterReplacement("AABABBA", 1))  # 4
print(characterReplacement("ABCD",    0))  # 1
print(characterReplacement("AAAA",    2))  # 4
test_harness(characterReplacement)

4
4
1
4
Test 1: PASSED | s='ABAB' k=2 | expected=4 | got=4
Test 2: PASSED | s='AABABBA' k=1 | expected=4 | got=4
Test 3: PASSED | s='A' k=0 | expected=1 | got=1
Test 4: PASSED | s='AAAA' k=2 | expected=4 | got=4
Test 5: PASSED | s='ABCD' k=0 | expected=1 | got=1
Test 6: PASSED | s='ABCD' k=3 | expected=4 | got=4
Test 7: PASSED | s='ABCDE' k=1 | expected=2 | got=2
Test 8: PASSED | s='AABABBA' k=0 | expected=2 | got=2
Test 9: PASSED | s='KRSCDCSONAJNHLBMDQGI...' k=4 | expected=7 | got=7

9/9 tests passed


In [ ]:
def characterReplacement(s: str, k: int) -> int:
    """
    Return longest substring of one char after <= k
    replacements.

    Sliding window: track char frequencies and the
    running max_count (never decremented). If
    window_size - max_count > k, slide left one step.
    Window size only ever grows -> result = final size.

    Time:  O(n) — each char enters and exits once
    Space: O(1) — at most 26 keys in count dict
    """
    pass


# Quick debug — run this cell while building
print(characterReplacement("ABAB",    2))  # 4
print(characterReplacement("AABABBA", 1))  # 4
print(characterReplacement("ABCD",    0))  # 1
print(characterReplacement("AAAA",    2))  # 4

In [ ]:
# Uncomment and run when solution is ready
# test_harness(characterReplacement)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — try all substrings | O(n²) | O(1) |
| Sliding window | O(n) | O(1) |

The window never actually shrinks below its current
size — left only moves when right does, so the
window slides forward in O(n) total steps.

## Real World Connection

At Citi, anomaly tolerance windows work on this
exact principle: given a time series of server
states (OK/WARN/CRIT), find the longest contiguous
period that stays within one alert level if up to
k deviations are allowed.
The sliding window computes this in one pass over
the telemetry stream, identifying stable operation
windows for SLA reporting even when individual
data points flicker.
In the ETL pipeline, the same pattern validates
data quality: the longest run of records that can
be considered clean if at most k are correctable
before the batch is rejected.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra